In [ ]:
%load_ext autoreload
%autoreload 2

import os
from src.constants import BASEDIR
os.chdir(BASEDIR)

from src.data.objects import Protein, ProteinDocument
from src.data.preprocessing import load_named_preprocessor
from src.data.sequence import sequence_identity
from src.models.inference import InterleavedInverseFoldingPromptBuilder, ProFamSampler
from src.models.utils import load_named_model

In [ ]:
protein = Protein.from_pdb(os.path.join(BASEDIR, "data/example_data/backbones/1qys.pdb"))
protein.sequence

TODO: pick some easy, high plddt targets from AFDB

TODO: figure out an easy way to instantiate preprocessor

TODO: if using random transforms - allow applying the pipeline multiple times to single target

In [ ]:
preprocessor = load_named_preprocessor("parquet_3di_with_interleaved_coords", overrides=["structure_tokens_col=null"])

In [ ]:
preprocessor.transform_fns

In [ ]:
model = load_named_model("llama_small", overrides=["model.embed_coords=True"])
sampler = ProFamSampler(
    "if_v1",
    model,
    prompt_builder=InterleavedInverseFoldingPromptBuilder(
        preprocessor=preprocessor,
        max_tokens=1536,
        representative_only=True,
    ),
    sampling_kwargs={"batch_size": 3, "temperature": 0.3},
    checkpoint_path=os.path.join(BASEDIR, "outputs/inverse_folding/neat-thunder/checkpoints/last.ckpt"),
    match_representative_length=True,
)

In [ ]:
samples, prompt = sampler.sample_seqs(
    ProteinDocument.from_proteins(
        [protein], representative_accession=protein.accession
    ), num_samples=3
)

In [ ]:
samples

In [ ]:
[sequence_identity(s, protein.sequence) for s in samples]